In [1]:
from rdkit import Chem
from rdkit.Chem import AllChem
import subprocess

def smiles_to_3d_xtb(smiles, out_xyz='molecule_opt.xyz'):
    # Создаем RDKit молекулу из SMILES
    mol = Chem.MolFromSmiles(smiles)
    mol = Chem.AddHs(mol)
    # Генерируем 3D конформер
    AllChem.EmbedMolecule(mol, AllChem.ETKDG())  
    AllChem.UFFOptimizeMolecule(mol)
    # Сохраняем временный xyz файл
    temp_xyz = 'temp.xyz'
    with open(temp_xyz, 'w') as f:
        f.write(Chem.MolToXYZBlock(mol))
    # Запускаем xtb оптимизацию с выходом в out_xyz
    subprocess.run([
        'xtb', temp_xyz, 
        '--opt', '--xyz', out_xyz
    ], check=True)
    print(f'Оптимизированная структура сохранена в {out_xyz}')

# Пример вызова
smiles_string = "C[C-]12[Ir+]3456([Cl][Ir+]789%10([Cl]3)([C-]3(C)[C]7(C)=[C]8(C)[C]9(=[C]%103C)C)Cl)([C]1(C)=[C]4(C)[C]5(=[C]26C)C)Cl"  # этанол, замените на нужный SMILES
smiles_to_3d_xtb(smiles_string)


[23:48:53] UFFTYPER: Unrecognized hybridization for atom: 2
[23:48:53] UFFTYPER: Unrecognized atom type: Ir (2)
[23:48:53] UFFTYPER: Unrecognized hybridization for atom: 4
[23:48:53] UFFTYPER: Unrecognized atom type: Ir (4)
[23:48:53] UFFTYPER: Unrecognized hybridization for atom: 2
[23:48:53] UFFTYPER: Unrecognized atom type: Ir (2)
[23:48:53] UFFTYPER: Unrecognized hybridization for atom: 4
[23:48:53] UFFTYPER: Unrecognized atom type: Ir (4)
[23:48:53] Could not triangle bounds smooth molecule.
[23:48:53] UFFTYPER: Unrecognized hybridization for atom: 2
[23:48:53] UFFTYPER: Unrecognized atom type: Ir (2)
[23:48:53] UFFTYPER: Unrecognized hybridization for atom: 4
[23:48:53] UFFTYPER: Unrecognized atom type: Ir (4)


ValueError: Bad Conformer Id

# Create df

In [1]:
import pandas as pd

df = pd.read_csv('/Users/egorilin/Desktop/MSU_AI/dataset.csv', sep=';')
df

,SMILES,Substance Identification: Reaxys Registry Number,Solvent (UV/VIS Spectroscopy),Absorption Maxima (UV/VIS) [nm],Ext./Abs. Coefficient [l·mol-1cm-1],References,Links to Reaxys
0,C[C-]12[Ir+]3456([Cl][Ir+]789%10([Cl]3)([C-]3(...,12591084,ethanol,NaN,NaN,"Article; Rojas-Luna, Raúl;Amaro-Gahete, Juan;...",https://www.reaxys.com/reaxys/secured/hopinto....
1,C[C-]12[Ir+]3456([Cl][Ir+]789%10([Cl]3)([C-]3(...,12591084,dichloromethane,242; 348,NaN,"Article; Chang, Elizabeth T.;Green, David B.;B...",https://www.reaxys.com/reaxys/secured/hopinto....
2,C[C-]12[Ir+]3456([Cl][Ir+]789%10([Cl]3)([C-]3(...,12591084,ethanol,227; 351,NaN,"Article; Ong, Kok Tong;Liu, Zhi-Qiang;Daud, Ad...",https://www.reaxys.com/reaxys/secured/hopinto....
3,C[C-]12[Ir+]3456([Cl][Ir+]789%10([Cl]3)([C-]3(...,12591084,NaN,417,NaN,"Article; Facchetti, Giorgio; Pellegrino, Sara;...",https://www.reaxys.com/reaxys/secured/hopinto....
4,[H][C]12=[C]3([H])CC[C]4([H])=[C]([H])(CC1)[Ir...,11993489,NaN,NaN,NaN,"Article; Riener, Korbinian; Meister, Teresa K....",https://www.reaxys.com/reaxys/secured/hopinto....
...,...,...,...,...,...,...,...
19163,CCCCN1C=CN2[C]1[Ru++]1345[C]6N(CCCC)C=CN6C6=CC...,17791074,acetonitrile,237; 266; 272; 283; 346; 385; 434,49890; 29070; 30330; 30200; 14950; 20400; 2510,"Article; Chung, Lai-Hon; Cho, Ka-Sin; England,...",https://www.reaxys.com/reaxys/secured/hopinto....
19164,C1C2=CC=CC=[N]2[Ru++]2345[N]6=CC=CC=C6C6=CC=CC...,17791345,dichloromethane,249; 295; 370; 425,28210; 31010; 13880; 11930,"Article; Weisser, Fritz; Plebst, Sebastian; Ho...",https://www.reaxys.com/reaxys/secured/hopinto....
19165,[O-]C(=O)C1=CC2=[N]3C(=C1)C1=[N](C=CC=C1)[Ru++...,18046639,water; methanol,485,27,"Article; Calabro, Rosemary; Glazer, Edith C.; ...",https://www.reaxys.com/reaxys/secured/hopinto....
19166,CC1=CC2=[N](C=C1)[Ru++]([Cl-])([Cl-])(C#[O])(C...,18047831,acetonitrile,310; 322; 362; 350,9500; 8800; 1900; 1800,"Article; Kubeil, Manja; Vernooij, Robbin R.; K...",https://www.reaxys.com/reaxys/secured/hopinto....


In [1]:
solvent_smiles = {
    '(2)H8-toluene': 'Cc1ccccc1',  # Требует уточнения для дейтерированной версии
    '1,1,2,2-tetrachloroethane': 'C(C(Cl)Cl)(Cl)Cl',
    '1,2-dichloro-benzene': 'Clc1ccccc1Cl',
    '1,2-dichloro-ethane': 'ClCCCl',
    '1,2-dimethoxyethane': 'COCCOC',
    '1,4-dioxane': 'O1CCOCC1',
    '1-methyl-pyrrolidin-2-one': 'CN1CCCC1=O',
    '2,2,2-trifluoroethanol': 'OCC(F)(F)F',
    '2,2,2-trifluoroethanol; water; trifluorormethanesulfonic acid': '[H]O[H]',
    '2-methyl-propan-2-ol': 'CC(C)(C)O',
    '2-methyltetrahydrofuran': 'CC1CCCO1',
    '4-(dicyanomethylene)-2-methyl-6-(p-dimethylaminostyryl)-4H-pyran': 'CN(C)C1=CC=C(C=CC2=CC(C=C(C)O2)=C(C#N)C#N)C=C1',
    '5,5-dimethyl-1,3-cyclohexadiene': 'CC1=CCC(C)=C1',
    'CCl4': 'ClC(Cl)(Cl)Cl',
    'CDCl3': '[2H]C(Cl)(Cl)Cl',
    'CH2Cl2': 'ClCCl',
    'CHCl3': 'ClC(Cl)Cl',
    'CS2': 'S=C=S',
    'Carbon tetrachloride': 'ClC(Cl)(Cl)Cl',
    'Dichlorofluoromethane': 'FC(Cl)Cl',
    'H2O': '[H]O[H]',
    'Isopropyl acetate': 'CC(=O)OCC(C)C',
    'MeCN': 'CC#N',
    'N,N-dimethyl acetamide': 'CC(=O)N(C)C',
    'N,N-dimethyl-acetamide': 'CC(=O)N(C)C',
    'N,N-dimethyl-formamide': 'CN(C)C=O',
    'N,N-dimethyl-formamide; aq. phosphate buffer': 'CN(C)C=O',
    'N,N-dimethyl-formamide; dichloromethane': 'CN(C)C=O',
    'N,N-dimethyl-formamide; trifluoroacetic acid': 'OC(=O)C(F)(F)F',
    'N,N-dimethyl-formamide; water': 'CN(C)C=O',
    'N,N-dimethyl-formamide; water monomer': 'CN(C)C=O',
    'N,N-dimethylformamide=DMF': 'CN(C)C=O',
    '[(2)H6]acetone': '[2H]C([2H])([2H])C(=O)C([2H])([2H])[2H]',
    '[D3]acetonitrile': '[2H]C([2H])([2H])C#N',
    'acetic acid': 'CC(=O)O',
    'acetone': 'CC(C)=O',
    'acetone; dimethyl sulfoxide; water': 'CS(=O)C',
    'acetone; hexane': 'CCCCCC',
    'acetone; water': 'CC(C)=O',
    'acetone; water monomer': 'CC(C)=O',
    'acetonitrile': 'CC#N',
    'acetonitrile; acetic acid': 'CC(=O)O',
    'acetonitrile; aq. phosphate buffer': 'CC#N',
    'acetonitrile; dimethyl sulfoxide': 'CS(=O)C',
    'acetonitrile; hydrogenchloride': 'CC#N',
    'acetonitrile; methanol': 'CO',
    'acetonitrile; tert-butyl alcohol': 'CC(C)(C)O',
    'acetonitrile; triethylamine': 'CCN(CC)CC',
    'acetonitrile; water': 'CC#N',
    'acetonitrile; water monomer': 'CC#N',
    'acidic aq. solution': 'CC#N',
    'acidic aq. solution; methanol': 'CO',
    'alkaline aq. solution': 'CC#N',
    'ammonia': '[NH3]',
    'aq. H2SO4': '[H]O[H]',
    'aq. HCl': '[H]O[H]',
    'aq. HClO4': '[H]O[H]',
    'aq. HF': '[H]O[H]',
    'aq. HNO3': '[H]O[H]',
    'aq. KOH': '[H]O[H]',
    'aq. NaOH': '[H]O[H]',
    'aq. acetate buffer': '[H]O[H]',
    'aq. acetate buffer; dimethyl sulfoxide': 'CS(=O)C',
    'aq. ammonia=NH3': '[H]O[H]',
    'aq. buffer': '[H]O[H]',
    'aq. buffer; N,N-dimethyl-formamide': 'CN(C)C=O',
    'aq. buffer; acetonitrile': 'CC#N',
    'aq. buffer; dimethyl sulfoxide': 'CS(=O)C',
    'aq. buffer; dimethyl sulfoxide; sodium chloride': 'CS(=O)C',
    'aq. buffer; ethanol': 'CCO',
    'aq. buffer; water; acetonitrile': '[H]O[H]',
    'aq. phosphate buffer': '[H]O[H]',
    'aq. phosphate buffer; (methylsulfinyl)methane': 'CS(=O)C',
    'aq. phosphate buffer; N,N-dimethyl-formamide': 'CN(C)C=O',
    'aq. phosphate buffer; acetonitrile': 'CC#N',
    'aq. phosphate buffer; dimethyl sulfoxide': 'CS(=O)C',
    'aq. phosphate buffer; sodium chloride': '[H]O[H]',
    'benzene': 'c1ccccc1',
    'benzene-d6': '[2H]c1c([2H])c([2H])c([2H])c([2H])c1[2H]',
    'benzonitrile': 'N#Cc1ccccc1',
    'butan-1-ol': 'CCCCO',
    'carbon dioxide': 'O=C=O',
    'chlorobenzene': 'Clc1ccccc1',
    'chloroform': 'ClC(Cl)Cl',
    'chloroform-d1': '[2H]C(Cl)(Cl)Cl',
    'chloroform; acetone': 'CC(C)=O',
    'chloroform; ethanol; acetic acid': 'CCO',
    'chloroform; hexane': 'CCCCCC',
    'chloroform; methanol': 'CO',
    'chloroform; triethylamine': 'CCN(CC)CC',
    'cyclohexane': 'C1CCCCC1',
    'decahydronaphthalene': 'C1CCC2CCCCC2C1',
    'decalin': 'C1CCC2CCCCC2C1',
    'dichloroethane=1,2-dichloroethane': 'ClCCCl',
    'dichloromethane': 'ClCCl',
    'dichloromethane-d2': '[2H]C([2H])(Cl)Cl',
    'dichloromethane; acetone': 'CC(C)=O',
    'dichloromethane; chloroform': 'ClC(Cl)Cl',
    'dichloromethane; dimethyl sulfoxide': 'CS(=O)C',
    'dichloromethane; methanol': 'CO',
    'dichloromethane; triethylamine': 'CCN(CC)CC',
    'dichloromethane; trifluoroacetic acid': 'OC(=O)C(F)(F)F',
    'diethyl ether': 'CCOCC',
    'diethyl ether; methylbutane; ethanol': 'CCC(C)C',
    'dimethyl sulfoxide': 'CS(=O)C',
    'dimethyl sulfoxide; N,N-dimethyl-formamide': 'CN(C)C=O',
    'dimethyl sulfoxide; aq. phosphate buffer': '[H]O[H]',
    'dimethyl sulfoxide; hydrogenchloride': 'Cl',
    'dimethyl sulfoxide; water': '[H]O[H]',
    'dimethyl sulfoxide; water monomer': '[H]O[H]',
    'dimethylformamide': 'CN(C)C=O',
    'dimethylsulfoxide': 'CS(=O)C',
    'dimethylsulfoxide-d6': '[2H]C([2H])([2H])S(=O)C([2H])([2H])[2H]',
    'dimethylsulfoxide=DMSO': 'CS(=O)C',
    'dioxane': 'O1CCOCC1',
    'ethanol': 'CCO',
    'ethanol; aq. phosphate buffer': '[H]O[H]',
    'ethanol; water': '[H]O[H]',
    'ethanol; water monomer': '[H]O[H]',
    'ethyl acetate': 'CC(=O)OCC',
    'ethylene glycol': 'OCCO',
    'fluorine': 'FF',
    'formamide': 'NC=O',
    'formic acid; water': '[H]O[H]',
    'freon 113=1,1,2-trichloro-trifluoroethane': 'FC(F)(F)C(Cl)(Cl)Cl',
    'glycerol': 'OCC(O)CO',
    'heptane': 'CCCCCCC',
    'hexane': 'CCCCCC',
    'hydrogen bromide': '[H]Br',
    'hydrogenchloride': 'Cl',
    'hydrogenchloride; water': '[H]O[H]',
    'hydrogenchloride; water monomer': '[H]O[H]',
    'hydrogenchloride; water; tetrahydrofuran': '[H]O[H]',
    'isooctane': 'CC(C)CC(C)(C)C',
    'isopropyl alcohol': 'CC(C)O',
    'liquid carbon dioxide': 'O=C=O',
    'm-xylene': 'Cc1cccc(C)c1',
    'm-xylene=m-xylol': 'Cc1cccc(C)c1',
    'methanol': 'CO',
    'methanol; 2-ethoxy-ethanol; water monomer': 'CCOCCO',
    'methanol; N,N-dimethyl-formamide': 'CN(C)C=O',
    'methanol; acetic acid': 'CC(=O)O',
    'methanol; aq. phosphate buffer': '[H]O[H]',
    'methanol; dimethyl sulfoxide': 'CS(=O)C',
    'methanol; water': '[H]O[H]',
    'methyl cyclohexane': 'CC1CCCCC1',
    'methyl cyclohexane; Trichloroethylene': 'ClC=C(Cl)Cl',
    'methylcyclohexane': 'CC1CCCCC1',
    'methylene chloride=methylene dichloride': 'ClCCl',
    'n-Pentane': 'CCCCC',
    'n-heptane': 'CCCCCCC',
    'neutral aq. solution': '[H]O[H]',
    'nitrobenzene': 'O=[N+]([O-])c1ccccc1',
    'nitromethane': 'C[N+](=O)[O-]',
    'octanol': 'CCCCCCCCO',
    'pentane': 'CCCCC',
    'phenyl cyanide': 'N#Cc1ccccc1',
    'potassium chloride; water': '[H]O[H]',
    'propan-1-ol': 'CCCO',
    'propan-2-ol': 'CC(C)O',
    'pyridine': 'c1ccncc1',
    'sodium chloride; water': '[H]O[H]',
    'sulfolane': 'O=S1(=O)CCCC1',
    'tert-butyl alcohol': 'CC(C)(C)O',
    'tetrachloromethane': 'ClC(Cl)(Cl)Cl',
    'tetrahydrofuran': 'C1CCCO1',
    'tetrahydrofuran-d8': '[2H]C1([2H])C([2H])([2H])C([2H])([2H])OC1([2H])[2H]',
    'tetrahydrofuran; N,N-dimethyl-formamide': 'CN(C)C=O',
    'tetrahydrofuran; benzene': 'c1ccccc1',
    'tetrahydrofuran; water': '[H]O[H]',
    'tetrahydrofuran; water monomer': '[H]O[H]',
    'toluene': 'Cc1ccccc1',
    'toluene; dichloromethane': 'ClCCl',
    'trifluorormethanesulfonic acid; water monomer; acetone': '[H]O[H]',
    'water': '[H]O[H]',
    'water monomer': '[H]O[H]',
    'water monomer; 2,2,2-trifluoroethanol': 'OCC(F)(F)F',
    'water monomer; dimethyl sulfoxide': 'CS(=O)C',
    'water monomer; ethanol': 'CCO',
    'water monomer; methanol': 'CO',
    'water monomer; tert-butyl alcohol': 'CC(C)(C)O',
    'water-d2': '[2H]O[2H]',
    'water-d2; acetonitrile': 'CC#N',
    'water; 2,2,2-trifluoroethanol': 'OCC(F)(F)F',
    'water; N,N-dimethyl-formamide': 'CN(C)C=O',
    'water; acetonitrile': 'CC#N',
    'water; dimethyl sulfoxide': 'CS(=O)C',
    'water; ethanol': 'CCO',
    'water; hydrogen cyanide': 'C#N',
    'water; methanol': 'CO',
    'water; tert-butyl alcohol': 'CC(C)(C)O',
    'water; tetrahydrofuran': 'C1CCCO1',
    'water; triethylamine': 'CCN(CC)CC',
    'water; trifluorormethanesulfonic acid': 'OS(=O)(=O)C(F)(F)F',
}

In [3]:
set(solvent_smiles.values())

{'C#N',
 'C(C(Cl)Cl)(Cl)Cl',
 'C1CCC2CCCCC2C1',
 'C1CCCCC1',
 'C1CCCO1',
 'CC#N',
 'CC(=O)N(C)C',
 'CC(=O)O',
 'CC(=O)OCC',
 'CC(=O)OCC(C)C',
 'CC(C)(C)O',
 'CC(C)=O',
 'CC(C)CC(C)(C)C',
 'CC(C)O',
 'CC1=CCC(C)=C1',
 'CC1CCCCC1',
 'CC1CCCO1',
 'CCC(C)C',
 'CCCCC',
 'CCCCCC',
 'CCCCCCC',
 'CCCCCCCCO',
 'CCCCO',
 'CCCO',
 'CCN(CC)CC',
 'CCO',
 'CCOCC',
 'CCOCCO',
 'CN(C)C1=CC=C(C=CC2=CC(C=C(C)O2)=C(C#N)C#N)C=C1',
 'CN(C)C=O',
 'CN1CCCC1=O',
 'CO',
 'COCCOC',
 'CS(=O)C',
 'C[N+](=O)[O-]',
 'Cc1cccc(C)c1',
 'Cc1ccccc1',
 'Cl',
 'ClC(Cl)(Cl)Cl',
 'ClC(Cl)Cl',
 'ClC=C(Cl)Cl',
 'ClCCCl',
 'ClCCl',
 'Clc1ccccc1',
 'Clc1ccccc1Cl',
 'FC(Cl)Cl',
 'FC(F)(F)C(Cl)(Cl)Cl',
 'FF',
 'N#Cc1ccccc1',
 'NC=O',
 'O1CCOCC1',
 'O=C=O',
 'O=S1(=O)CCCC1',
 'O=[N+]([O-])c1ccccc1',
 'OC(=O)C(F)(F)F',
 'OCC(F)(F)F',
 'OCC(O)CO',
 'OCCO',
 'OS(=O)(=O)C(F)(F)F',
 'S=C=S',
 '[2H]C(Cl)(Cl)Cl',
 '[2H]C([2H])(Cl)Cl',
 '[2H]C([2H])([2H])C#N',
 '[2H]C([2H])([2H])C(=O)C([2H])([2H])[2H]',
 '[2H]C([2H])([2H])S(=O)C([2H])([2H

In [ ]:
import pandas as pd
import numpy as np

def find_max_absorption(row):
    """
    Находит значение поглощения с максимальным коэффициентом экстинкции
    или медианное значение, если коэффициенты отсутствуют.
    """
    absorption = row['Absorption Maxima (UV/VIS) [nm]']
    coefficient = row['Ext./Abs. Coefficient [l·mol-1cm-1]']
    
    # Если нет данных о поглощении, возвращаем NaN
    if pd.isna(absorption) or absorption == '':
        return np.nan
    
    # Разбиваем значения поглощения
    absorption_values = [float(x.strip()) for x in str(absorption).split(';') if x.strip()]
    
    # Фильтруем значения в диапазоне 380-700 нм
    filtered_absorption = [val for val in absorption_values if 380 <= val <= 700]
    
    # Если нет значений в диапазоне, возвращаем NaN
    if not filtered_absorption:
        return np.nan
    
    # Проверяем наличие коэффициентов экстинкции
    if pd.isna(coefficient) or coefficient == '':
        # Если коэффициентов нет, возвращаем медиану
        return np.median(filtered_absorption)
    
    # Разбиваем коэффициенты
    coefficient_values = []
    for x in str(coefficient).split(';'):
        x = x.strip()
        if x:
            try:
                coefficient_values.append(float(x))
            except ValueError:
                coefficient_values.append(np.nan)
    
    # Создаем пары (поглощение, коэффициент) только для значений в диапазоне
    pairs = []
    for i, abs_val in enumerate(absorption_values):
        if 350 <= abs_val <= 700:
            coef = coefficient_values[i] if i < len(coefficient_values) else np.nan
            pairs.append((abs_val, coef))
    
    # Если все коэффициенты NaN, возвращаем медиану
    valid_pairs = [(a, c) for a, c in pairs if not pd.isna(c)]
    
    if not valid_pairs:
        return np.median(filtered_absorption)
    
    # Находим поглощение с максимальным коэффициентом
    max_pair = max(valid_pairs, key=lambda x: x[1])
    return max_pair[0]

Проверка на первых 10 строках:

Строка 1:
  Absorption Maxima: nan
  Coefficients: nan
  Selected Absorption: nan
----------------------------------------------------------------------------------------------------

Строка 2:
  Absorption Maxima: 242; 348
  Coefficients: nan
  Selected Absorption: nan
----------------------------------------------------------------------------------------------------

Строка 3:
  Absorption Maxima: 227; 351
  Coefficients: nan
  Selected Absorption: nan
----------------------------------------------------------------------------------------------------

Строка 4:
  Absorption Maxima: 417
  Coefficients: nan
  Selected Absorption: 417.0
----------------------------------------------------------------------------------------------------

Строка 5:
  Absorption Maxima: nan
  Coefficients: nan
  Selected Absorption: nan
----------------------------------------------------------------------------------------------------

Строка 6:
  Absorption Maxima: 359
 

In [ ]:
import pandas as pd
import numpy as np
from openbabel import pybel
import os
import re

def sanitize_filename(smiles, max_length=50):
    """
    Преобразует SMILES в безопасное имя файла.
    Убирает специальные символы и ограничивает длину.
    """
    # Заменяем специальные символы на подчеркивания
    safe_name = re.sub(r'[^\w\-]', '_', smiles)
    # Убираем множественные подчеркивания
    safe_name = re.sub(r'_+', '_', safe_name)
    # Ограничиваем длину
    if len(safe_name) > max_length:
        safe_name = safe_name[:max_length]
    return safe_name

def extract_ligands_from_smiles(smiles, row_index):
    """
    Извлекает лиганды из SMILES комплекса металла.
    Возвращает список SMILES лигандов и сохраняет xyz файлы.
    """
    # Список металлов (добавлен родий Rh=45)
    metals_atomic_nums = {26, 28, 29, 30, 44, 45, 46, 77, 78, 27, 25, 24, 42}
    
    try:
        # Создаем молекулу из SMILES
        mol = pybel.readstring("smi", smiles)
        
        # Генерируем 3D координаты
        mol.make3D()
        
        # Удаляем металлы в обратном порядке индексов
        metal_atoms = [atom for atom in mol.atoms if atom.atomicnum in metals_atomic_nums]
        metal_atoms_sorted = sorted(metal_atoms, key=lambda a: a.idx, reverse=True)
        
        for atom in metal_atoms_sorted:
            mol.OBMol.DeleteAtom(atom.OBAtom)
        
        # Разбиваем на связные фрагменты
        fragments = mol.OBMol.Separate()
        
        ligands = []
        # Создаем безопасное имя из SMILES
        safe_smiles = sanitize_filename(smiles)
        
        for i, frag_obmol in enumerate(fragments):
            ligand_mol = pybel.Molecule(frag_obmol)
            ligand_smiles = ligand_mol.write("smi").strip()
            
            # Имя файла: safe_smiles_lig_number.xyz
            ligand_xyz = f"ligand_files/{safe_smiles}_lig_{i+1}.xyz"
            
            # Создаем директорию если не существует
            os.makedirs("ligand_files", exist_ok=True)
            
            # Сохраняем xyz файл
            ligand_mol.write("xyz", ligand_xyz, overwrite=True)
            
            ligands.append(ligand_smiles)
            print(f"  Лиганд {i+1}: {ligand_smiles[:50]}... -> {ligand_xyz}")
        
        return ligands
    
    except Exception as e:
        print(f"  Ошибка при обработке SMILES: {e}")
        return []

def process_dataframe(df, solvent_smiles):
    """
    Обрабатывает датафрейм: добавляет SMILES растворителя, 
    выбранное поглощение и лиганды.
    """
    # Создаем новый датафрейм
    new_df = pd.DataFrame()
    
    # Копируем основные колонки
    new_df['Metal_Complex_SMILES'] = df['SMILES']
    new_df['Reaxys_Registry_Number'] = df['Substance Identification: Reaxys Registry Number']
    new_df['Solvent_Name'] = df['Solvent (UV/VIS Spectroscopy)']
    
    # Добавляем SMILES растворителя из словаря
    new_df['Solvent_SMILES'] = new_df['Solvent_Name'].map(solvent_smiles)
    
    # Добавляем выбранное поглощение (уже посчитанное)
    if 'Selected_Absorption' in df.columns:
        new_df['Selected_Absorption'] = df['Selected_Absorption']
    else:
        # Если еще не посчитано, применяем функцию
        new_df['Selected_Absorption'] = df.apply(find_max_absorption, axis=1)
    
    # Инициализируем колонки для лигандов
    new_df['lig_1'] = np.nan
    new_df['lig_2'] = np.nan
    new_df['lig_3'] = np.nan
    new_df['lig_4'] = np.nan
    new_df['lig_5'] = np.nan
    
    # Обрабатываем каждую строку
    print("Обработка комплексов и извлечение лигандов:")
    print("="*100)
    
    for idx, row in df.iterrows():
        print(f"\nСтрока {idx + 1}:")
        smiles = row['SMILES']
        
        if pd.isna(smiles) or smiles == '':
            print("  Пропуск: пустой SMILES")
            continue
        
        # Извлекаем лиганды
        ligands = extract_ligands_from_smiles(smiles, idx)
        
        # Записываем лиганды в соответствующие колонки
        for i, ligand in enumerate(ligands):
            if i < 5:  # Максимум 5 лигандов
                new_df.at[idx, f'lig_{i+1}'] = ligand
        
        print(f"  Всего найдено лигандов: {len(ligands)}")
    
    return new_df

# Применяем обработку
print("Создание нового датафрейма...")
new_df = process_dataframe(df, solvent_smiles)

# Выводим результат
print("\n" + "="*100)
print("РЕЗУЛЬТАТ:")
print("="*100)
print(new_df.head(10))

# Сохраняем в CSV
new_df.to_csv('processed_complexes.csv', index=False, sep=';')
print("\nДатафрейм сохранен в 'processed_complexes.csv'")

# Выводим статистику
print("\nСтатистика:")
print(f"Всего строк: {len(new_df)}")
print(f"Строк с растворителем: {new_df['Solvent_SMILES'].notna().sum()}")
print(f"Строк с выбранным поглощением: {new_df['Selected_Absorption'].notna().sum()}")
print(f"Строк с лигандами: {new_df['lig_1'].notna().sum()}")

In [ ]:
import pandas as pd
import numpy as np
from openbabel import pybel
import os
from rdkit import Chem
from rdkit.Chem import AllChem

def extract_ligands_from_smiles(smiles, row_index):
    """
    Извлекает лиганды из SMILES комплекса металла.
    Возвращает список SMILES лигандов и сохраняет xyz файлы.
    """
    # Список металлов (добавлен родий Rh=45)
    metals_atomic_nums = {26, 28, 29, 30, 44, 45, 46, 77, 78, 27, 25, 24, 42}
    
    try:
        # Создаем молекулу из SMILES
        mol = pybel.readstring("smi", smiles)
        
        # Генерируем 3D координаты
        mol.make3D()
        
        # Удаляем металлы в обратном порядке индексов
        metal_atoms = [atom for atom in mol.atoms if atom.atomicnum in metals_atomic_nums]
        metal_atoms_sorted = sorted(metal_atoms, key=lambda a: a.idx, reverse=True)
        
        for atom in metal_atoms_sorted:
            mol.OBMol.DeleteAtom(atom.OBAtom)
        
        # Разбиваем на связные фрагменты
        fragments = mol.OBMol.Separate()
        
        ligands = []
        for i, frag_obmol in enumerate(fragments):
            ligand_mol = pybel.Molecule(frag_obmol)
            ligand_smiles = ligand_mol.write("smi").strip()
            
            # Имя файла: row_index_ligand_number.xyz
            ligand_xyz = f"ligand_files_2/{row_index}_lig_{i+1}.xyz"
            
            # Создаем директорию если не существует
            os.makedirs("ligand_files_2", exist_ok=True)
            
            # Сохраняем xyz файл
            ligand_mol.write("xyz", ligand_xyz, overwrite=True)
            
            ligands.append(ligand_smiles)
            print(f"  Лиганд {i+1}: {ligand_smiles[:50]}... -> {ligand_xyz}")
        
        return ligands
    
    except Exception as e:
        print(f"  Ошибка при обработке SMILES: {e}")
        return []

def process_dataframe(df, solvent_smiles):
    """
    Обрабатывает датафрейм: добавляет SMILES растворителя, 
    выбранное поглощение и лиганды.
    """
    # Создаем новый датафрейм
    new_df = pd.DataFrame()
    
    # Копируем основные колонки
    new_df['Metal_Complex_SMILES'] = df['SMILES']
    new_df['Reaxys_Registry_Number'] = df['Substance Identification: Reaxys Registry Number']
    new_df['Solvent_Name'] = df['Solvent (UV/VIS Spectroscopy)']
    
    # Добавляем SMILES растворителя из словаря
    new_df['Solvent_SMILES'] = new_df['Solvent_Name'].map(solvent_smiles)
    
    # Добавляем выбранное поглощение (уже посчитанное)
    if 'Selected_Absorption' in df.columns:
        new_df['Selected_Absorption'] = df['Selected_Absorption']
    else:
        # Если еще не посчитано, применяем функцию
        new_df['Selected_Absorption'] = df.apply(find_max_absorption, axis=1)
    
    # Инициализируем колонки для лигандов
    new_df['lig_1'] = np.nan
    new_df['lig_2'] = np.nan
    new_df['lig_3'] = np.nan
    new_df['lig_4'] = np.nan
    new_df['lig_5'] = np.nan
    
    # Обрабатываем каждую строку
    print("Обработка комплексов и извлечение лигандов:")
    print("="*100)
    
    for idx, row in df.iterrows():
        print(f"\nСтрока {idx + 1}:")
        smiles = row['SMILES']
        
        if pd.isna(smiles) or smiles == '':
            print("  Пропуск: пустой SMILES")
            continue
        
        # Извлекаем лиганды
        ligands = extract_ligands_from_smiles(smiles, idx)
        
        # Записываем лиганды в соответствующие колонки
        for i, ligand in enumerate(ligands):
            if i < 5:  # Максимум 5 лигандов
                new_df.at[idx, f'lig_{i+1}'] = ligand
        
        print(f"  Всего найдено лигандов: {len(ligands)}")
    
    return new_df

# Применяем обработку
print("Создание нового датафрейма...")
new_df = process_dataframe(df, solvent_smiles)

# Выводим результат
print("\n" + "="*100)
print("РЕЗУЛЬТАТ:")
print("="*100)
print(new_df.head(10))

# Сохраняем в CSV
new_df.to_csv('processed_complexes_2.csv', index=False, sep=';')
print("\nДатафрейм сохранен в 'processed_complexes_2.csv'")

# Выводим статистику
print("\nСтатистика:")
print(f"Всего строк: {len(new_df)}")
print(f"Строк с растворителем: {new_df['Solvent_SMILES'].notna().sum()}")
print(f"Строк с выбранным поглощением: {new_df['Selected_Absorption'].notna().sum()}")
print(f"Строк с лигандами: {new_df['lig_1'].notna().sum()}")

# Opt Solvents

In [11]:
from rdkit import Chem
from rdkit.Chem import AllChem
import subprocess
import os
import re

def sanitize_filename(name, max_length=50):
    """
    Преобразует название в безопасное имя файла.
    """
    # Заменяем специальные символы на подчеркивания
    safe_name = re.sub(r'[^\w\-]', '_', name)
    # Убираем множественные подчеркивания
    safe_name = re.sub(r'_+', '_', safe_name)
    # Убираем подчеркивания в начале и конце
    safe_name = safe_name.strip('_')
    # Ограничиваем длину
    if len(safe_name) > max_length:
        safe_name = safe_name[:max_length]
    return safe_name

def smiles_to_3d_xtb(smiles, out_xyz='molecule_opt.xyz', keep_temp=False):
    """
    Создает 3D структуру из SMILES и оптимизирует её с помощью xtb.
    
    Parameters:
    -----------
    smiles : str
        SMILES строка молекулы
    out_xyz : str
        Имя выходного файла для оптимизированной структуры
    keep_temp : bool
        Сохранять ли временные файлы
    
    Returns:
    --------
    bool : True если успешно, False если ошибка
    """
    try:
        # Создаем RDKit молекулу из SMILES
        mol = Chem.MolFromSmiles(smiles)
        
        if mol is None:
            print(f"  ❌ Ошибка: невозможно создать молекулу из SMILES: {smiles}")
            return False
        
        mol = Chem.AddHs(mol)
        
        # Генерируем 3D конформер
        result = AllChem.EmbedMolecule(mol, AllChem.ETKDG())
        
        if result != 0:
            print(f"  ⚠️ Предупреждение: проблема при генерации 3D структуры")
        
        AllChem.UFFOptimizeMolecule(mol)
        
        # Сохраняем временный xyz файл
        temp_xyz = 'temp_solvent.xyz'
        with open(temp_xyz, 'w') as f:
            f.write(Chem.MolToXYZBlock(mol))
        
        # Запускаем xtb оптимизацию
        result = subprocess.run([
            'xtb', temp_xyz, 
            '--opt'
        ], 
        capture_output=True, 
        text=True,
        check=False)
        
        # Проверяем, что xtb создал файл xtbopt.xyz (стандартное имя)
        if os.path.exists('xtbopt.xyz'):
            # Переименовываем в нужное имя
            os.rename('xtbopt.xyz', out_xyz)
            print(f"  ✓ Оптимизация завершена: {out_xyz}")
            success = True
        else:
            print(f"  ❌ Ошибка: xtb не создал оптимизированный файл")
            success = False
        
        # Удаляем временные файлы xtb
        temp_files = ['xtbrestart', 'xtbtopo.mol', 'wbo', 'charges', 
                      'gfnff_topo', '.xtboptok', 'xtbopt.log', temp_xyz]
        if not keep_temp:
            for f in temp_files:
                if os.path.exists(f):
                    os.remove(f)
        
        return success
        
    except Exception as e:
        print(f"  ❌ Ошибка: {e}")
        return False

def optimize_all_solvents(solvent_smiles, output_dir='optimized_solvents'):
    """
    Оптимизирует все растворители из словаря solvent_smiles.
    
    Parameters:
    -----------
    solvent_smiles : dict
        Словарь {название_растворителя: SMILES}
    output_dir : str
        Директория для сохранения оптимизированных структур
    """
    # Создаем директорию для сохранения
    os.makedirs(output_dir, exist_ok=True)
    
    print("="*100)
    print(f"ОПТИМИЗАЦИЯ РАСТВОРИТЕЛЕЙ")
    print("="*100)
    print(f"Всего растворителей в словаре: {len(solvent_smiles)}")
    print()
    
    success_count = 0
    fail_count = 0
    skip_count = 0
    
    for i, (solvent_name, smiles) in enumerate(solvent_smiles.items(), 1):
        print(f"[{i}/{len(solvent_smiles)}] {solvent_name}")
        print(f"  SMILES: {smiles}")
        
        # Создаем безопасное имя файла
        safe_name = sanitize_filename(solvent_name)
        out_file = os.path.join(output_dir, f"{safe_name}.xyz")
        
        # Проверяем, не оптимизирован ли уже
        if os.path.exists(out_file):
            print(f"  ✓ Уже существует: {out_file}")
            skip_count += 1
            print()
            continue
        
        # Оптимизируем структуру
        success = smiles_to_3d_xtb(smiles, out_file)
        
        if success:
            success_count += 1
        else:
            fail_count += 1
        
        print()
    
    print("="*100)
    print("СТАТИСТИКА ОПТИМИЗАЦИИ:")
    print(f"  ✓ Успешно оптимизировано: {success_count}")
    print(f"  ⏭️  Пропущено (уже есть): {skip_count}")
    print(f"  ❌ Ошибок оптимизации: {fail_count}")
    print(f"  📁 Файлы сохранены в: {output_dir}/")
    print("="*100)

# ИСПОЛЬЗОВАНИЕ:
# Запускаем оптимизацию всех растворителей из словаря
optimize_all_solvents(solvent_smiles)

ОПТИМИЗАЦИЯ РАСТВОРИТЕЛЕЙ
Всего растворителей в словаре: 194

[1/194] (2)H8-toluene
  SMILES: Cc1ccccc1
  ✓ Оптимизация завершена: optimized_solvents/2_H8-toluene.xyz

[2/194] 1,1,2,2-tetrachloroethane
  SMILES: C(C(Cl)Cl)(Cl)Cl
  ✓ Оптимизация завершена: optimized_solvents/1_1_2_2-tetrachloroethane.xyz

[3/194] 1,2-dichloro-benzene
  SMILES: Clc1ccccc1Cl
  ✓ Оптимизация завершена: optimized_solvents/1_2-dichloro-benzene.xyz

[4/194] 1,2-dichloro-ethane
  SMILES: ClCCCl
  ✓ Оптимизация завершена: optimized_solvents/1_2-dichloro-ethane.xyz

[5/194] 1,2-dimethoxyethane
  SMILES: COCCOC
  ✓ Оптимизация завершена: optimized_solvents/1_2-dimethoxyethane.xyz

[6/194] 1,4-dioxane
  SMILES: O1CCOCC1
  ✓ Оптимизация завершена: optimized_solvents/1_4-dioxane.xyz

[7/194] 1-methyl-pyrrolidin-2-one
  SMILES: CN1CCCC1=O
  ✓ Оптимизация завершена: optimized_solvents/1-methyl-pyrrolidin-2-one.xyz

[8/194] 2,2,2-trifluoroethanol
  SMILES: OCC(F)(F)F
  ✓ Оптимизация завершена: optimized_solvents/2_2_2-

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem
import subprocess
import os
import pandas as pd
from tqdm import tqdm

def smiles_to_3d_xtb(smiles, out_xyz='molecule_opt.xyz', keep_temp=False):
    """
    Создает 3D структуру из SMILES и оптимизирует её с помощью xtb.
    
    Parameters:
    -----------
    smiles : str
        SMILES строка молекулы
    out_xyz : str
        Имя выходного файла для оптимизированной структуры
    keep_temp : bool
        Сохранять ли временные файлы
    
    Returns:
    --------
    bool : True если успешно, False если ошибка
    """
    try:
        # Создаем RDKit молекулу из SMILES
        mol = Chem.MolFromSmiles(smiles)
        
        if mol is None:
            return False
        
        mol = Chem.AddHs(mol)
        
        # Генерируем 3D конформер
        result = AllChem.EmbedMolecule(mol, AllChem.ETKDG())
        
        if result != 0:
            # Пробуем без ETKDG если не получилось
            result = AllChem.EmbedMolecule(mol, randomSeed=42)
            if result != 0:
                return False
        
        AllChem.UFFOptimizeMolecule(mol)
        
        # Сохраняем временный xyz файл
        temp_xyz = 'temp_complex.xyz'
        with open(temp_xyz, 'w') as f:
            f.write(Chem.MolToXYZBlock(mol))
        
        # Запускаем xtb оптимизацию
        result = subprocess.run([
            'xtb', temp_xyz, 
            '--opt'
        ], 
        capture_output=True, 
        text=True,
        check=False)
        
        # Проверяем, что xtb создал файл xtbopt.xyz (стандартное имя)
        if os.path.exists('xtbopt.xyz'):
            # Переименовываем в нужное имя
            os.rename('xtbopt.xyz', out_xyz)
            success = True
        else:
            success = False
        
        # Удаляем временные файлы xtb
        temp_files = ['xtbrestart', 'xtbtopo.mol', 'wbo', 'charges', 
                      'gfnff_topo', '.xtboptok', 'xtbopt.log', temp_xyz]
        if not keep_temp:
            for f in temp_files:
                if os.path.exists(f):
                    os.remove(f)
        
        return success
        
    except Exception as e:
        return False

def optimize_complexes_from_df(df, smiles_column='SMILES', output_dir='optimized_complexes'):
    """
    Оптимизирует все молекулы из датафрейма.
    
    Parameters:
    -----------
    df : DataFrame
        Датафрейм с колонкой SMILES
    smiles_column : str
        Название колонки со SMILES
    output_dir : str
        Директория для сохранения оптимизированных структур
    
    Returns:
    --------
    DataFrame : Обновленный датафрейм с колонкой путей к файлам
    """
    # Создаем директорию для сохранения
    os.makedirs(output_dir, exist_ok=True)
    
    print("="*100)
    print(f"ОПТИМИЗАЦИЯ КОМПЛЕКСОВ ИЗ ДАТАФРЕЙМА")
    print("="*100)
    print(f"Всего строк в датафрейме: {len(df)}")
    print()
    
    # Добавляем колонку для путей к файлам
    df['Complex_XYZ_File'] = None
    
    success_count = 0
    fail_count = 0
    skip_count = 0
    empty_count = 0
    
    # Используем tqdm для прогресс-бара
    for idx in tqdm(range(len(df)), desc="Оптимизация комплексов", unit="молекула"):
        row = df.iloc[idx]
        smiles = row[smiles_column]
        
        # Проверяем, есть ли SMILES
        if pd.isna(smiles) or smiles == '':
            empty_count += 1
            continue
        
        # Имя файла: порядковый номер (начиная с 0 или с 1 - как хотите)
        out_file = os.path.join(output_dir, f"complex_{idx}.xyz")
        
        # Проверяем, не оптимизирован ли уже
        if os.path.exists(out_file):
            df.at[idx, 'Complex_XYZ_File'] = out_file
            skip_count += 1
            continue
        
        # Оптимизируем структуру
        success = smiles_to_3d_xtb(smiles, out_file)
        
        if success:
            df.at[idx, 'Complex_XYZ_File'] = out_file
            success_count += 1
        else:
            fail_count += 1
    
    print()
    print("="*100)
    print("СТАТИСТИКА ОПТИМИЗАЦИИ:")
    print(f"  ✓ Успешно оптимизировано: {success_count}")
    print(f"  ⏭️  Пропущено (уже есть): {skip_count}")
    print(f"  ❌ Ошибок оптимизации: {fail_count}")
    print(f"  ⚠️  Пустых SMILES: {empty_count}")
    print(f"  📁 Файлы сохранены в: {output_dir}/")
    print("="*100)
    
    return df

# ИСПОЛЬЗОВАНИЕ:
# Оптимизируем все комплексы из датафрейма
df = optimize_complexes_from_df(df, smiles_column='SMILES', output_dir='optimized_complexes')

# # Сохраняем обновленный датафрейм
# df.to_csv('processed_complexes_with_xyz.csv', index=False, sep=';')

# Выводим примеры
print("\nПримеры обработанных строк:")
# print(df[['SMILES', 'Complex_XYZ_File']].head(10))

Оптимизация комплексов:  10%|▉         | 1852/19168 [8:04:18<230:35:11, 47.94s/молекула][07:53:39] UFFTYPER: Unrecognized atom type: Ir6 (15)
[07:53:39] UFFTYPER: Unrecognized atom type: Ir6 (15)
Оптимизация комплексов:  10%|▉         | 1853/19168 [8:04:24<174:19:45, 36.25s/молекула][07:53:45] UFFTYPER: Unrecognized atom type: Ir6 (23)
[07:53:46] UFFTYPER: Unrecognized atom type: Ir6 (23)


In [4]:
from rdkit import Chem
from rdkit.Chem import AllChem
import subprocess
import os
from tqdm import tqdm
import pandas as pd
from rdkit import RDLogger

# Отключаем предупреждения RDKit
RDLogger.DisableLog('rdApp.*')

def smiles_to_3d_xtb(smiles, out_xyz='molecule_opt.xyz', keep_temp=False):
    """
    Создает 3D структуру из SMILES и оптимизирует её с помощью xtb.
    Пропускает UFF оптимизацию для металлосодержащих комплексов.
    
    Parameters:
    -----------
    smiles : str
        SMILES строка молекулы
    out_xyz : str
        Имя выходного файла для оптимизированной структуры
    keep_temp : bool
        Сохранять ли временные файлы
    
    Returns:
    --------
    bool : True если успешно, False если ошибка
    """
    try:
        # Создаем RDKit молекулу из SMILES
        mol = Chem.MolFromSmiles(smiles)
        
        if mol is None:
            return False
        
        mol = Chem.AddHs(mol)
        
        # Список атомных номеров металлов
        metals = {21, 22, 23, 24, 25, 26, 27, 28, 29, 30,  # 3d металлы
                  39, 40, 41, 42, 43, 44, 45, 46, 47, 48,  # 4d металлы
                  57, 72, 73, 74, 75, 76, 77, 78, 79, 80}  # 5d металлы и лантаниды
        
        # Проверяем, есть ли металлы в молекуле
        has_metals = any(atom.GetAtomicNum() in metals for atom in mol.GetAtoms())
        
        # Генерируем 3D конформер
        params = AllChem.ETKDGv3()
        params.randomSeed = 42
        result = AllChem.EmbedMolecule(mol, params)
        
        if result != 0:
            # Пробуем базовый метод если ETKDG не работает
            result = AllChem.EmbedMolecule(mol, randomSeed=42)
            if result != 0:
                # Пробуем без рандомного сида
                result = AllChem.EmbedMolecule(mol)
                if result != 0:
                    return False
        
        # UFF оптимизация ТОЛЬКО если нет металлов
        if not has_metals:
            try:
                AllChem.UFFOptimizeMolecule(mol, maxIters=200)
            except:
                # Если UFF не работает, продолжаем без оптимизации
                pass
        
        # Сохраняем временный xyz файл
        temp_xyz = 'temp_complex.xyz'
        with open(temp_xyz, 'w') as f:
            xyz_block = Chem.MolToXYZBlock(mol)
            if xyz_block:
                f.write(xyz_block)
            else:
                return False
        
        # Запускаем xtb оптимизацию
        result = subprocess.run([
            'xtb', temp_xyz, 
            '--opt',
            '--gbsa', 'h2o'  # Неявный растворитель (можно убрать если не нужен)
        ], 
        capture_output=True, 
        text=True,
        check=False,
        timeout=300)  # Таймаут 5 минут на одну молекулу
        
        # Проверяем, что xtb создал файл xtbopt.xyz
        if os.path.exists('xtbopt.xyz'):
            os.rename('xtbopt.xyz', out_xyz)
            success = True
        else:
            success = False
        
        # Удаляем временные файлы xtb
        temp_files = ['xtbrestart', 'xtbtopo.mol', 'wbo', 'charges', 
                      'gfnff_topo', '.xtboptok', 'xtbopt.log', temp_xyz,
                      'xtb.out', 'xtbhess.xyz', 'g98.out', 'xtbopt.coord']
        if not keep_temp:
            for f in temp_files:
                if os.path.exists(f):
                    try:
                        os.remove(f)
                    except:
                        pass
        
        return success
        
    except subprocess.TimeoutExpired:
        return False
    except Exception as e:
        return False

def optimize_complexes_from_df(df, smiles_column='SMILES', output_dir='optimized_complexes'):
    """
    Оптимизирует все молекулы из датафрейма с прогресс-баром.
    
    Parameters:
    -----------
    df : DataFrame
        Датафрейм с колонкой SMILES
    smiles_column : str
        Название колонки со SMILES
    output_dir : str
        Директория для сохранения оптимизированных структур
    
    Returns:
    --------
    DataFrame : Обновленный датафрейм с колонкой путей к файлам
    """
    # Создаем директорию для сохранения
    os.makedirs(output_dir, exist_ok=True)
    
    print("="*100)
    print(f"ОПТИМИЗАЦИЯ КОМПЛЕКСОВ ИЗ ДАТАФРЕЙМА")
    print("="*100)
    print(f"Всего строк в датафрейме: {len(df)}")
    print(f"Металлы будут обрабатываться без UFF оптимизации (только xtb)")
    print()
    
    # Добавляем колонку для путей к файлам
    df['Complex_XYZ_File'] = None
    
    success_count = 0
    fail_count = 0
    skip_count = 0
    empty_count = 0
    
    # Используем tqdm для прогресс-бара
    for idx in tqdm(range(len(df)), desc="Оптимизация комплексов", unit="молекула"):
        row = df.iloc[idx]
        smiles = row[smiles_column]
        
        # Проверяем, есть ли SMILES
        if pd.isna(smiles) or smiles == '':
            empty_count += 1
            continue
        
        # Имя файла: порядковый номер
        out_file = os.path.join(output_dir, f"complex_{idx}.xyz")
        
        # Проверяем, не оптимизирован ли уже
        if os.path.exists(out_file):
            df.at[idx, 'Complex_XYZ_File'] = out_file
            skip_count += 1
            continue
        
        # Оптимизируем структуру
        success = smiles_to_3d_xtb(smiles, out_file)
        
        if success:
            df.at[idx, 'Complex_XYZ_File'] = out_file
            success_count += 1
        else:
            fail_count += 1
    
    print()
    print("="*100)
    print("СТАТИСТИКА ОПТИМИЗАЦИИ:")
    print(f"  ✓ Успешно оптимизировано: {success_count}")
    print(f"  ⏭️  Пропущено (уже есть): {skip_count}")
    print(f"  ❌ Ошибок оптимизации: {fail_count}")
    print(f"  ⚠️  Пустых SMILES: {empty_count}")
    print(f"  📁 Файлы сохранены в: {output_dir}/")
    print("="*100)
    
    return df

# ИСПОЛЬЗОВАНИЕ:
# Оптимизируем все комплексы из датафрейма
df = optimize_complexes_from_df(df, smiles_column='SMILES', output_dir='optimized_complexes')

# # Сохраняем обновленный датафрейм
# df.to_csv('processed_complexes_with_xyz.csv', index=False, sep=';')

# print("\nПримеры обработанных строк:")
# print(df[['SMILES', 'Complex_XYZ_File']].head(10))

ОПТИМИЗАЦИЯ КОМПЛЕКСОВ ИЗ ДАТАФРЕЙМА
Всего строк в датафрейме: 19168
Металлы будут обрабатываться без UFF оптимизации (только xtb)



Оптимизация комплексов:   0%|          | 12/19168 [02:45<96:49:41, 18.20s/молекула] 

: 

: 

In [2]:
from rdkit import Chem
from rdkit.Chem import AllChem
import os
import pandas as pd
from tqdm import tqdm

def smiles_to_3d_rdkit(smiles, out_xyz='molecule_opt.xyz', max_attempts=2):
    """
    Создает 3D структуру из SMILES и оптимизирует её с помощью RDKit.
    
    Parameters:
    -----------
    smiles : str
        SMILES строка молекулы
    out_xyz : str
        Имя выходного файла для оптимизированной структуры
    max_attempts : int
        Максимальное количество попыток генерации конформера
    
    Returns:
    --------
    bool : True если успешно, False если ошибка
    """
    try:
        # Создаем RDKit молекулу из SMILES
        mol = Chem.MolFromSmiles(smiles)
        
        if mol is None:
            return False
        
        mol = Chem.AddHs(mol)
        
        # Пытаемся сгенерировать 3D конформер с несколькими попытками
        success = False
        for attempt in range(max_attempts):
            result = AllChem.EmbedMolecule(mol, AllChem.ETKDG())
            
            if result == 0:  # Успешно
                break
            else:
                # Пробуем с другим методом
                result = AllChem.EmbedMolecule(mol, randomSeed=42 + attempt)
                if result == 0:
                    break
        
        if result != 0:
            return False
        
        # Оптимизируем молекулу с UFF
        try:
            result = AllChem.UFFOptimizeMolecule(mol, maxIters=2)
            if result != 0:  # Если UFF не сработал, пробуем MMFF
                try:
                    AllChem.MMFFOptimizeMolecule(mol, maxIters=2)
                except:
                    pass  # Если и MMFF не сработал, используем текущую структуру
        except:
            pass  # Если UFF не доступен, используем текущую структуру
        
        # Сохраняем XYZ файл
        with open(out_xyz, 'w') as f:
            f.write(Chem.MolToXYZBlock(mol))
        
        return True
        
    except Exception as e:
        return False

def optimize_complexes_from_df(df, smiles_column='SMILES', output_dir='optimized_complexes_rdkit', start_index=0):
    """
    Оптимизирует все молекулы из датафрейма с помощью RDKit.
    
    Parameters:
    -----------
    df : DataFrame
        Датафрейм с колонкой SMILES
    smiles_column : str
        Название колонки со SMILES
    output_dir : str
        Директория для сохранения оптимизированных структур
    start_index : int
        Порядковый номер строки, с которой начинается расчет
    
    Returns:
    --------
    DataFrame : Обновленный датафрейм с колонкой путей к файлам
    """
    # Создаем директорию для сохранения
    os.makedirs(output_dir, exist_ok=True)
    
    print("="*100)
    print(f"ОПТИМИЗАЦИЯ КОМПЛЕКСОВ ИЗ ДАТАФРЕЙМА (RDKit)")
    print("="*100)
    print(f"Всего строк в датафрейме: {len(df)}")
    print(f"Начальный индекс: {start_index}")
    print()
    
    # Добавляем колонку для путей к файлам
    df['Complex_XYZ_File'] = None
    
    success_count = 0
    fail_count = 0
    skip_count = 0
    empty_count = 0
    
    # Используем tqdm для прогресс-бара, начиная с заданного индекса
    for idx in tqdm(range(start_index, len(df)), desc="Оптимизация комплексов", unit="молекула"):
        row = df.iloc[idx]
        smiles = row[smiles_column]
        
        # Проверяем, есть ли SMILES
        if pd.isna(smiles) or smiles == '':
            empty_count += 1
            continue
        
        # Имя файла: порядковый номер
        out_file = os.path.join(output_dir, f"complex_{idx}.xyz")
        
        # Проверяем, не оптимизирован ли уже
        if os.path.exists(out_file):
            df.at[idx, 'Complex_XYZ_File'] = out_file
            skip_count += 1
            continue
        
        # Оптимизируем структуру
        success = smiles_to_3d_rdkit(smiles, out_file)
        
        if success:
            df.at[idx, 'Complex_XYZ_File'] = out_file
            success_count += 1
        else:
            fail_count += 1
    
    print()
    print("="*100)
    print("СТАТИСТИКА ОПТИМИЗАЦИИ:")
    print(f"  ✓ Успешно оптимизировано: {success_count}")
    print(f"  ⏭️  Пропущено (уже есть): {skip_count}")
    print(f"  ❌ Ошибок оптимизации: {fail_count}")
    print(f"  ⚠️  Пустых SMILES: {empty_count}")
    print(f"  📁 Файлы сохранены в: {output_dir}/")
    print("="*100)
    
    return df

# Укажите начальный индекс здесь
i = 10000 # измените это значение на нужный начальный индекс

import pandas as pd

df = pd.read_csv('/Users/egorilin/Desktop/MSU_AI/dataset.csv', sep=';')

# Оптимизируем все комплексы из датафрейма, начиная с индекса i
df = optimize_complexes_from_df(df, smiles_column='SMILES', output_dir='optimized_complexes_rdkit', start_index=i)

# # Сохраняем обновленный датафрейм
# df.to_csv('processed_complexes_with_xyz.csv', index=False, sep=';')

# Выводим примеры
print("\nПримеры обработанных строк:")
print(df[['SMILES', 'Complex_XYZ_File']].iloc[i:i+10])

[16:47:45] UFFTYPER: Unrecognized charge state for atom: 6
[16:47:45] UFFTYPER: Unrecognized atom type: Rh6 (19)
[16:47:45] UFFTYPER: Unrecognized charge state for atom: 20
[16:47:45] UFFTYPER: Unrecognized charge state for atom: 24
[16:47:45] UFFTYPER: Unrecognized charge state for atom: 69
Оптимизация комплексов:  76%|███████▌  | 6974/9168 [6:38:26<1:48:05,  2.96s/молекула][16:47:45] UFFTYPER: Unrecognized charge state for atom: 6
[16:47:45] UFFTYPER: Unrecognized atom type: Rh6 (19)
[16:47:45] UFFTYPER: Unrecognized charge state for atom: 20
[16:47:45] UFFTYPER: Unrecognized charge state for atom: 24
[16:47:45] UFFTYPER: Unrecognized charge state for atom: 69
[16:47:48] UFFTYPER: Unrecognized charge state for atom: 6
[16:47:48] UFFTYPER: Unrecognized atom type: Rh6 (19)
[16:47:48] UFFTYPER: Unrecognized charge state for atom: 20
[16:47:48] UFFTYPER: Unrecognized charge state for atom: 24
[16:47:48] UFFTYPER: Unrecognized charge state for atom: 69
Оптимизация комплексов:  76%|███████


СТАТИСТИКА ОПТИМИЗАЦИИ:
  ✓ Успешно оптимизировано: 4405
  ⏭️  Пропущено (уже есть): 0
  ❌ Ошибок оптимизации: 4428
  ⚠️  Пустых SMILES: 335
  📁 Файлы сохранены в: optimized_complexes_rdkit/

Примеры обработанных строк:
                                                  SMILES  \
10000  CCCCC(CCCC)C1=CC=[N](C=C1)[Pt++]([Cl-])([Cl-])...   
10001  CCC1=CC=C(C=C1)[N]1=CC2=[C-](C=C3C4=C5C(C=CC=C...   
10002  C[S](C)(=O)[Pt++]12[C-]3=C(SC=C3)C3=[N]1C(=CC=...   
10003  FC1=C[C-]2=C(C3=[N](C=CC=C3)[Pt++]22[O-]C(=CC(...   
10004  FC1=C(F)C(F)=[C-](C(F)=C1F)[Pt++]1([C-]2=C3C4=...   
10005  FC(F)(F)C1=CC=C(C=C1)[C]1#[C-]2[Pt++]1134[C-]5...   
10006  COC1=CC=C(C=C1)C1=N[N-](C2=CC=C(C)C=C2)[Pt@++]...   
10007  CC1=CC(C)=[O][Pt++]2([O-]1)[C-]1=C(C=CC=C1)C1=...   
10008  CC(C)(C)C1=CC(=[O][Pt++]2([O-]1)[C-]1=C(C3=[N]...   
10009  CC(C)(C)C1=CC(=[O][Pt++]2([O-]1)[C-]1=CC=CC=C1...   

                                  Complex_XYZ_File  
10000                                         None  
10001  optim